# 04 · Aggregations

Summarising data: totals, averages, counts per group, pivots. If you know SQL `GROUP BY`, this will feel familiar.

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)

## Aggregating a whole table

In [ ]:
orders.agg(
    F.count("*").alias("n_orders"),
    F.sum("quantity").alias("total_qty"),
    F.round(F.avg("quantity"), 3).alias("avg_qty"),
    F.min("order_ts").alias("first"),
    F.max("order_ts").alias("last"),
    F.countDistinct("customer_id").alias("n_customers"),
).show(truncate=False)

### `count("*")` vs `count(col)`
`count("*")` counts rows. `count("col")` counts **non-null** values in that column.

In [ ]:
customers.agg(F.count("*").alias("rows"), F.count("country").alias("with_country")).show()

## `groupBy` + `agg`
`groupBy(cols)` splits rows into groups; `agg(...)` computes one row per group.

In [ ]:
orders.groupBy("status").count().show()

(orders
 .groupBy("status")
 .agg(F.count("*").alias("n"),
      F.sum("quantity").alias("total_qty"),
      F.round(F.avg("quantity"), 2).alias("avg_qty"))
 .orderBy(F.desc("n"))
 .show())

In [ ]:
# group by several columns, including a computed one
(orders
 .groupBy(F.month("order_ts").alias("month"), "status")
 .count()
 .orderBy("month", "status")
 .show(6))

Shortcuts like `.groupBy('status').sum('quantity')` exist but give ugly column names like `sum(quantity)` — prefer `agg` + `alias`.

## Filtering after aggregating (SQL's `HAVING`)
Just `filter` the aggregated DataFrame:

In [ ]:
(orders.groupBy("customer_id").agg(F.count("*").alias("n_orders"))
       .filter(F.col("n_orders") > 38)
       .orderBy(F.desc("n_orders"))
       .show())

## Collecting values into lists

In [ ]:
(orders.filter(F.col("customer_id") == 1)
       .groupBy("customer_id")
       .agg(F.collect_list("product_id").alias("all_products"),
            F.sort_array(F.collect_set("product_id")).alias("unique_products"))
       .show(truncate=False))

## Pivot
Turns the *values* of one column into *columns*. Passing the list of values (`pivot("status", [...])`)
saves Spark an extra pass to discover them.

In [ ]:
(orders
 .groupBy(F.month("order_ts").alias("month"))
 .pivot("status", ["completed", "cancelled", "returned"])
 .count()
 .orderBy("month")
 .show())

## Subtotals with `rollup` and `cube`
`rollup("a", "b")` gives groups (a, b), subtotals per a, and a grand total (shown as nulls).

In [ ]:
(orders.rollup("status", F.year("order_ts").alias("year"))
       .agg(F.sum("quantity").alias("qty"))
       .orderBy("status", "year")
       .show())

## Approximate counts on big data
`countDistinct` is exact but expensive at scale. `approx_count_distinct` (HyperLogLog) is far cheaper and
usually within a couple of percent.

In [ ]:
orders.agg(F.countDistinct("customer_id"), F.approx_count_distinct("customer_id")).show()

### Exercise 1
Create `qty_by_status` with columns **`status`, `total_qty`** (the sum of quantity per status).

In [ ]:
qty_by_status = None  # TODO

In [ ]:
# Check — run me after your answer
assert sorted(qty_by_status.columns) == ["status", "total_qty"]
assert qty_by_status.filter(F.col("status") == "completed").first().total_qty == 8905
print("Correct!")

### Exercise 2
Create `top_customers`: the **5** customers with the most orders, columns **`customer_id`, `n_orders`**, sorted by `n_orders` descending.

In [ ]:
top_customers = None  # TODO

In [ ]:
# Check — run me after your answer
assert top_customers.columns == ["customer_id", "n_orders"]
assert [r.n_orders for r in top_customers.collect()] == [40, 39, 38, 38, 38]
print("Correct!")

### Exercise 3
Create `monthly` with one row per month (1–12) and columns **`month`, `n_orders`, `n_customers`** (distinct customers), sorted by month.

In [ ]:
monthly = None  # TODO

In [ ]:
# Check — run me after your answer
rows = monthly.collect()
assert monthly.columns == ["month", "n_orders", "n_customers"]
assert len(rows) == 12 and rows[0].month == 1
assert (rows[0].n_orders, rows[0].n_customers) == (404, 157)
assert sum(r.n_orders for r in rows) == 5000
print("Correct!")

### Exercise 4
Create `status_pivot`: one row per month with the order **count** for each status as its own column (`completed`, `cancelled`, `returned`).

In [ ]:
status_pivot = None  # TODO

In [ ]:
# Check — run me after your answer
assert status_pivot.count() == 12
assert {"cancelled", "completed", "returned"} <= set(status_pivot.columns)
assert status_pivot.agg(F.sum("completed")).first()[0] == 4035
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/04_aggregations.ipynb`](../solutions/04_aggregations.ipynb).